# Explorando o warehouse da GeloDados

Este notebook é para explorar os dados **depois** que o pipeline já rodou — não substitui o `scripts/run_pipeline.py`, complementa.

**Antes de rodar este notebook:**
```bash
python scripts/run_pipeline.py
```

Se `warehouse/workshop.duckdb` não existir, as células abaixo vão dar erro.

Dependências extras (além do `requirements.txt` principal): `pip install -r requirements-notebook.txt`

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 4.5)

con = duckdb.connect("../warehouse/workshop.duckdb", read_only=True)

## 1. O dado bruto (raw) — antes de qualquer limpeza

`raw.vendas_historico` é o espelho fiel do CSV exportado pelo sistema de vendas — com toda a sujeira que um export real costuma ter.

In [ ]:
con.execute("SELECT * FROM raw.vendas_historico LIMIT 8").df()

Quanta sujeira exatamente? Vamos medir antes de decidir como limpar.

In [ ]:
con.execute("""
    SELECT
        COUNT(*) AS total_linhas,
        SUM(CASE WHEN data_venda LIKE '__/__/____' THEN 1 ELSE 0 END) AS datas_formato_br,
        SUM(CASE WHEN genero_cliente = '' OR genero_cliente IS NULL THEN 1 ELSE 0 END) AS genero_faltando,
        SUM(CASE WHEN idade_cliente = '' OR idade_cliente IS NULL THEN 1 ELSE 0 END) AS idade_faltando,
        SUM(CASE WHEN CAST(unidades_vendidas AS INTEGER) > 50 THEN 1 ELSE 0 END) AS outliers_suspeitos
    FROM raw.vendas_historico
""").df()

## 2. Staging — depois da limpeza e reconciliação

`stg_vendas` já uniu `raw.vendas_historico` + `raw.vendas_recentes` (schema drift resolvido), tipou as colunas certo e removeu duplicatas exatas.

Repare que o total de linhas é menor que `4400 + 260` — as duplicatas propositais saíram.

In [ ]:
raw_total = con.execute("SELECT COUNT(*) FROM raw.vendas_historico").fetchone()[0]
raw_total += con.execute("SELECT COUNT(*) FROM raw.vendas_recentes").fetchone()[0]
stg_total = con.execute("SELECT COUNT(*) FROM stg_vendas").fetchone()[0]

print(f"raw (histórico + recentes): {raw_total} linhas")
print(f"stg_vendas (depois do dedup): {stg_total} linhas")
print(f"duplicatas removidas: {raw_total - stg_total}")

O outlier de 500 unidades não foi apagado — ficou sinalizado em `unidades_suspeitas`, pra quem for analisar decidir o que fazer com ele.

In [ ]:
con.execute("SELECT * FROM stg_vendas WHERE unidades_suspeitas").df()

## 3. Mart — a pergunta de negócio: o clima influencia as vendas?

`mart_vendas_clima` já está no grão dia/loja, juntado com o clima do dia. Dá pra responder a pergunta central do case direto daqui.

In [ ]:
df = con.execute("SELECT * FROM mart_vendas_clima").df()
df.head()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df["temp_max_c"], df["receita_total_brl"], alpha=0.4, s=18)
ax.set_xlabel("Temperatura máxima do dia (°C)")
ax.set_ylabel("Receita do dia na loja (R$)")
ax.set_title("Receita diária por loja vs. temperatura máxima")
plt.show()

In [ ]:
receita_por_faixa = df.groupby("faixa_temperatura")["receita_total_brl"].sum().reindex(["< 20C", "20-28C", "> 28C"])

fig, ax = plt.subplots()
receita_por_faixa.plot(kind="bar", ax=ax, color="#f2994a")
ax.set_xlabel("Faixa de temperatura")
ax.set_ylabel("Receita total (R$)")
ax.set_title("Receita total por faixa de temperatura")
plt.xticks(rotation=0)
plt.show()

## 4. Por loja: o efeito é igual em todo lugar?

Curitiba e Porto Alegre (mais frias) vs. Fortaleza (mais quente) devem contar histórias bem diferentes aqui.

In [ ]:
pivot = df.pivot_table(
    index="cidade", columns="faixa_temperatura", values="receita_total_brl", aggfunc="sum"
).reindex(columns=["< 20C", "20-28C", "> 28C"])

pivot.plot(kind="bar", figsize=(9, 5))
plt.ylabel("Receita total (R$)")
plt.title("Receita por cidade e faixa de temperatura")
plt.xticks(rotation=0)
plt.legend(title="Temperatura")
plt.show()

## 5. Stretch: qual sabor vende mais em cada temperatura?

In [ ]:
sabor = con.execute("SELECT * FROM mart_sabor_temperatura").df()
top_por_faixa = (
    sabor.sort_values("unidades_vendidas", ascending=False)
    .groupby("faixa_temperatura")
    .head(3)
    .sort_values(["faixa_temperatura", "unidades_vendidas"], ascending=[True, False])
)
top_por_faixa

## 6. Stretch: quem são os melhores clientes?

In [ ]:
top_clientes = con.execute("""
    SELECT nome_cliente, loja_favorita, qtd_compras, receita_total_brl, ticket_medio_brl
    FROM mart_clientes
    ORDER BY receita_total_brl DESC
    LIMIT 10
""").df()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_clientes["nome_cliente"][::-1], top_clientes["receita_total_brl"][::-1], color="#46c2d9")
ax.set_xlabel("Receita total (R$)")
ax.set_title("Top 10 clientes por receita")
plt.show()

## 7. Agora é sua vez

Use `stg_vendas` e os marts para responder (escreva o SQL ou pandas você mesmo):

1. Qual é o ticket médio por forma de pagamento (`pix`, `cartao`, `dinheiro`)?
2. Existe alguma loja com clientes mais fiéis — poucos clientes distintos fazendo muitas compras, em vez de muitos clientes ocasionais?
3. Qual é o sabor mais vendido **por loja** (em vez de por temperatura)?

In [ ]:
# Exercício 1: ticket médio por forma de pagamento
# seu código aqui

In [ ]:
# Exercício 2: lojas com clientes mais fiéis
# dica: COUNT(DISTINCT cliente_id) vs COUNT(*) por loja em stg_vendas
# seu código aqui

In [ ]:
# Exercício 3: sabor mais vendido por loja
# seu código aqui

---
Para continuar depois da aula: veja o `PLAN.md` e o `README.md` na raiz do repositório.